In [1]:
import pandas as pd 
import numpy as np 
import tensorflow as tf 

In [2]:
df = pd.read_csv('Dataset\IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [6]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(df['review'],
                                        df['sentiment'],
                                        test_size = 0.3,
                                        random_state=42)

In [7]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words = 10000,
    oov_token = '<OOV>'
)

In [ ]:
tokenizer.fit_on_texts(X_train) #Basically Ranks each word based on its frequency 

In [ ]:
tokenizer.word_index

In [9]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [10]:
X_train_padded = tf.keras.preprocessing.sequence.pad_sequences(
    X_train_seq,
    maxlen = 200,
    padding = 'post',
    truncating = 'post'
)

X_test_padded = tf.keras.preprocessing.sequence.pad_sequences(
    X_test_seq,
    maxlen = 200,
    padding = 'post',
    truncating = 'post'
)

In [12]:
model = tf.keras.models.Sequential([

    tf.keras.layers.Embedding(
        input_dim = 100000,
        output_dim = 64,
        input_length = 200
    ),

    tf.keras.layers.LSTM(64),

    tf.keras.layers.Dense(64, activation='relu'),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation='sigmoid')

])

d:\LSTM\.venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(
    optimizer = 'adam',
    loss='binary_crossentropy',
    metrics = ['accuracy']
)

In [15]:
history = model.fit(

    X_train_padded,

    y_train,

    validation_data=(X_test_padded, y_test),

    epochs=5,

    batch_size=32
)

Epoch 1/5
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 160s 144ms/step - accuracy: 0.5437 - loss: 0.6850 - val_accuracy: 0.5707 - val_loss: 0.6718
Epoch 2/5
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 173s 159ms/step - accuracy: 0.5743 - loss: 0.6716 - val_accuracy: 0.5969 - val_loss: 0.6479
Epoch 3/5
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 163s 149ms/step - accuracy: 0.5947 - loss: 0.6491 - val_accuracy: 0.8037 - val_loss: 0.4557
Epoch 4/5
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 156s 143ms/step - accuracy: 0.8593 - loss: 0.3526 - val_accuracy: 0.8682 - val_loss: 0.3161
Epoch 5/5
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 165s 151ms/step - accuracy: 0.9062 - loss: 0.2540 - val_accuracy: 0.8695 - val_loss: 0.3075


In [16]:
model.save('LSTM.keras')

In [29]:
def sentiment_analysis(sentence):

    embed = tokenizer.texts_to_sequences(sentence)

    embed_pad = tf.keras.preprocessing.sequence.pad_sequences(
    embed,
    maxlen = 200,
    padding = 'post',
    truncating = 'post'
    )

    score = model.predict(embed_pad)[0][0]

    return 'Positive' if score > 0.5 else 'Negative' , f'Score : {score}' 

    

In [37]:
print(sentiment_analysis('The movie was soooo awesome i will never watch it again'))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
('Negative', 'Score : 0.2942233979701996')
